# Experiment 3: Regression Analysis using Linear and Regularized Models

**Objective:**  
Implement and evaluate Linear, Ridge, Lasso, and Elastic Net regression models for predicting continuous loan sanction amounts. Perform hyperparameter optimization using 5-fold cross-validation, inspect feature coefficient shrinkage, and analyze bias-variance trade-offs.

## Step 1: Environment Setup & Library Imports

In [ ]:
import os
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
os.makedirs('figures', exist_ok=True)

# Set plot appearance
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 10
print('Libraries successfully imported.')

## Step 2: Data Loading & Preprocessing Workflow

- Load `train.csv` dataset
- Filter out invalid target instances
- Impute missing numeric values using column medians
- Impute missing categorical attributes using column modes
- Apply One-Hot Encoding and Standard Scaling

In [ ]:
# Load dataset
dataset_filename = 'train.csv'
raw_dataframe = pd.read_csv(dataset_filename)

# Drop identifier columns
unused_cols = ['Customer ID', 'Name', 'Property ID']
clean_df = raw_dataframe.drop(columns=[c for c in unused_cols if c in raw_dataframe.columns], errors='ignore')

# Target selection & cleanup
target_name = 'Loan Sanction Amount (USD)'
clean_df = clean_df.dropna(subset=[target_name])
clean_df = clean_df[clean_df[target_name] > 0].copy()

# Separate numeric and categorical predictor columns
numeric_features = clean_df.select_dtypes(include=[np.number]).columns.tolist()
numeric_features.remove(target_name)
categorical_features = clean_df.select_dtypes(include=['object', 'category']).columns.tolist()

# Apply Median Imputation for numerical features
median_imputer = SimpleImputer(strategy='median')
clean_df[numeric_features] = median_imputer.fit_transform(clean_df[numeric_features])

# Apply Mode Imputation for categorical features
mode_imputer = SimpleImputer(strategy='most_frequent')
clean_df[categorical_features] = mode_imputer.fit_transform(clean_df[categorical_features])

print(f'Processed dataset size: {clean_df.shape[0]} rows, {clean_df.shape[1]} columns')

## Step 3 & 4: Exploratory Data Analysis & Visualizations

Generate and save required exploratory visualizations:

In [ ]:
# 1. Target Distribution Plot
fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(clean_df[target_name], kde=True, color='#2b5c8f', bins=40, ax=ax, edgecolor='white', alpha=0.7)
ax.set_title('Target Distribution: Loan Sanction Amount (USD)', pad=12, fontweight='bold')
ax.set_xlabel('Loan Sanction Amount (USD)')
ax.set_ylabel('Frequency')
ax.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
fig.savefig('figures/target_distribution.png', dpi=300)
fig.savefig('target_distribution.png', dpi=300)
plt.show()

# 2. Key Predictors Scatter Plots
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
scatter_pairs = [
    ('Loan Amount Request (USD)', 'Loan Request vs Sanction Amount', '#1f77b4', axes[0, 0]),
    ('Income (USD)', 'Applicant Income vs Sanction Amount', '#ff7f0e', axes[0, 1]),
    ('Credit Score', 'Credit Score vs Sanction Amount', '#2ca02c', axes[1, 0]),
    ('Property Price', 'Property Price vs Sanction Amount', '#9467bd', axes[1, 1])
]
for col, title, color, ax in scatter_pairs:
    if col in clean_df.columns:
        ax.scatter(clean_df[col], clean_df[target_name], alpha=0.35, color=color, edgecolors='none', s=18)
        ax.set_title(title, fontweight='bold', fontsize=11)
        ax.set_xlabel(col)
        ax.set_ylabel('Sanction Amount (USD)')
        ax.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
fig.savefig('figures/scatter_plots.png', dpi=300)
fig.savefig('scatter_plots.png', dpi=300)
plt.show()

# 3. Correlation Matrix Heatmap
fig, ax = plt.subplots(figsize=(10, 8))
corr_matrix = clean_df[numeric_features + [target_name]].corr()
sns.heatmap(corr_matrix, cmap='Spectral_r', annot=True, fmt='.2f', linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8}, annot_kws={'size': 8})
ax.set_title('Correlation Matrix of Numerical Attributes', pad=12, fontweight='bold')
plt.tight_layout()
fig.savefig('figures/correlation_heatmap.png', dpi=300)
fig.savefig('correlation_heatmap.png', dpi=300)
plt.show()

## Step 5: Feature Matrix Encoding, Standardization & Train-Test Split

In [ ]:
# Encoding categorical attributes
X_unscaled = pd.get_dummies(clean_df.drop(columns=[target_name]), columns=categorical_features, drop_first=True)
y_labels = clean_df[target_name].values

# Feature Scaling
standard_scaler = StandardScaler()
X_scaled = standard_scaler.fit_transform(X_unscaled)
feature_names = X_unscaled.columns.tolist()

# 80/20 Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_labels, test_size=0.2, random_state=42
)
print(f'Training shape: {X_train.shape}, Testing shape: {X_test.shape}')

## Step 6 & 7: Model Fitting (Linear, Ridge, Lasso, Elastic Net) & Hyperparameter Tuning (5-Fold CV)

In [ ]:
def evaluate_regression(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    return {'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'R2': r2}

# 1. Baseline OLS Linear Regression
ols_reg = LinearRegression()
t_start = time.time()
ols_reg.fit(X_train, y_train)
ols_time = time.time() - t_start
ols_perf = evaluate_regression(y_test, ols_reg.predict(X_test))
ols_perf['Time'] = ols_time

# 2. Ridge Hyperparameter Tuning
ridge_params = {'alpha': [0.01, 0.1, 1.0, 10.0, 100.0]}
ridge_cv = GridSearchCV(Ridge(random_state=42), ridge_params, cv=5, scoring='r2', n_jobs=-1)
t_start = time.time()
ridge_cv.fit(X_train, y_train)
ridge_time = time.time() - t_start
best_ridge = ridge_cv.best_estimator_
ridge_perf = evaluate_regression(y_test, best_ridge.predict(X_test))
ridge_perf['Time'] = ridge_time

# 3. Lasso Hyperparameter Tuning
lasso_params = {'alpha': [0.001, 0.01, 0.1, 1.0, 10.0]}
lasso_cv = GridSearchCV(Lasso(random_state=42, max_iter=5000), lasso_params, cv=5, scoring='r2', n_jobs=-1)
t_start = time.time()
lasso_cv.fit(X_train, y_train)
lasso_time = time.time() - t_start
best_lasso = lasso_cv.best_estimator_
lasso_perf = evaluate_regression(y_test, best_lasso.predict(X_test))
lasso_perf['Time'] = lasso_time

# 4. Elastic Net Hyperparameter Tuning
elastic_params = {'alpha': [0.01, 0.1, 1.0, 10.0], 'l1_ratio': [0.2, 0.5, 0.8]}
elastic_cv = GridSearchCV(ElasticNet(random_state=42, max_iter=5000), elastic_params, cv=5, scoring='r2', n_jobs=-1)
t_start = time.time()
elastic_cv.fit(X_train, y_train)
elastic_time = time.time() - t_start
best_elastic = elastic_cv.best_estimator_
elastic_perf = evaluate_regression(y_test, best_elastic.predict(X_test))
elastic_perf['Time'] = elastic_time

## Step 8 & 9: Results Summary, Cross-Validation & Model Evaluation Tables

In [ ]:
# Table 1: Hyperparameter Tuning Summary
tuning_df = pd.DataFrame({
    'Model': ['Ridge Regression', 'Lasso Regression', 'Elastic Net Regression'],
    'Search Method': ['GridSearchCV', 'GridSearchCV', 'GridSearchCV'],
    'Best Parameters': [str(ridge_cv.best_params_), str(lasso_cv.best_params_), str(elastic_cv.best_params_)],
    'Best CV R2': [f'{ridge_cv.best_score_:.4f}', f'{lasso_cv.best_score_:.4f}', f'{elastic_cv.best_score_:.4f}']
})
print('=== Table 1: Hyperparameter Tuning Summary ===')
display(tuning_df)

# Table 2: 5-Fold Cross-Validation Performance
cv_folds = KFold(n_splits=5, shuffle=True, random_state=42)
cv_metrics = ['neg_mean_absolute_error', 'neg_mean_squared_error', 'r2']
cv_rows = []
models_dict = {'Linear Regression': ols_reg, 'Ridge Regression': best_ridge, 'Lasso Regression': best_lasso, 'Elastic Net Regression': best_elastic}
for name, m in models_dict.items():
    res = cross_validate(m, X_scaled, y_labels, cv=cv_folds, scoring=cv_metrics, n_jobs=-1)
    mae_cv = -res['test_neg_mean_absolute_error'].mean()
    mse_cv = -res['test_neg_mean_squared_error'].mean()
    cv_rows.append({
        'Model': name,
        'MAE': f'${mae_cv:,.2f}',
        'MSE': f'${mse_cv:,.2f}',
        'RMSE': f'${np.sqrt(mse_cv):,.2f}',
        'R2': f'{res["test_r2"].mean():.4f}'
    })
cv_table = pd.DataFrame(cv_rows)
print('=== Table 2: 5-Fold Cross-Validation Performance ===')
display(cv_table)

# Table 3: Test Set Performance Comparison
test_df = pd.DataFrame([
    {'Model': 'Linear Regression', 'MAE': f'${ols_perf["MAE"]:,.2f}', 'MSE': f'${ols_perf["MSE"]:,.2f}', 'RMSE': f'${ols_perf["RMSE"]:,.2f}', 'R2': f'{ols_perf["R2"]:.4f}', 'Execution Time': f'{ols_perf["Time"]:.4f} s'},
    {'Model': 'Ridge Regression', 'MAE': f'${ridge_perf["MAE"]:,.2f}', 'MSE': f'${ridge_perf["MSE"]:,.2f}', 'RMSE': f'${ridge_perf["RMSE"]:,.2f}', 'R2': f'{ridge_perf["R2"]:.4f}', 'Execution Time': f'{ridge_perf["Time"]:.4f} s'},
    {'Model': 'Lasso Regression', 'MAE': f'${lasso_perf["MAE"]:,.2f}', 'MSE': f'${lasso_perf["MSE"]:,.2f}', 'RMSE': f'${lasso_perf["RMSE"]:,.2f}', 'R2': f'{lasso_perf["R2"]:.4f}', 'Execution Time': f'{lasso_perf["Time"]:.4f} s'},
    {'Model': 'Elastic Net Regression', 'MAE': f'${elastic_perf["MAE"]:,.2f}', 'MSE': f'${elastic_perf["MSE"]:,.2f}', 'RMSE': f'${elastic_perf["RMSE"]:,.2f}', 'R2': f'{elastic_perf["R2"]:.4f}', 'Execution Time': f'{elastic_perf["Time"]:.4f} s'}
])
print('=== Table 3: Test Set Performance Comparison ===')
display(test_df)

## Step 9 (Continued): Model Behavior & Diagnostic Plots

In [ ]:
# 4. Coefficient Comparison Chart
coef_comparison_df = pd.DataFrame({
    'Feature': feature_names,
    'Linear': ols_reg.coef_,
    'Ridge': best_ridge.coef_,
    'Lasso': best_lasso.coef_,
    'Elastic Net': best_elastic.coef_
})
top_feats = coef_comparison_df.iloc[np.argsort(-np.abs(ols_reg.coef_))[:8]]
fig, ax = plt.subplots(figsize=(11, 6))
x_pos = np.arange(len(top_feats))
w = 0.2
ax.bar(x_pos - 1.5*w, top_feats['Linear'], width=w, label='Linear', color='#3498db')
ax.bar(x_pos - 0.5*w, top_feats['Ridge'], width=w, label='Ridge', color='#e67e22')
ax.bar(x_pos + 0.5*w, top_feats['Lasso'], width=w, label='Lasso', color='#2ecc71')
ax.bar(x_pos + 1.5*w, top_feats['Elastic Net'], width=w, label='Elastic Net', color='#9b59b6')
ax.set_xticks(x_pos)
ax.set_xticklabels(top_feats['Feature'], rotation=35, ha='right', fontsize=9)
ax.set_ylabel('Coefficient Magnitude')
ax.set_title('Coefficient Shrinkage Across Regression Models', fontweight='bold', pad=12)
ax.legend()
ax.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
fig.savefig('figures/coef_comparison.png', dpi=300)
fig.savefig('coef_comparison.png', dpi=300)
plt.show()

# 5. Predicted vs Actual Plot
fig, ax = plt.subplots(figsize=(7, 6))
y_ridge_pred = best_ridge.predict(X_test)
ax.scatter(y_test, y_ridge_pred, alpha=0.35, color='#2c3e50', edgecolors='none', s=20)
min_val, max_val = min(y_test.min(), y_ridge_pred.min()), max(y_test.max(), y_ridge_pred.max())
ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Ideal Fit (y = x)')
ax.set_title('Predicted vs. Actual Loan Sanction Amount', fontweight='bold', pad=12)
ax.set_xlabel('Actual Loan Sanction Amount (USD)')
ax.set_ylabel('Predicted Loan Sanction Amount (USD)')
ax.legend()
ax.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
fig.savefig('figures/pred_vs_actual.png', dpi=300)
fig.savefig('pred_vs_actual.png', dpi=300)
plt.show()

# 6. Residual Plot
fig, ax = plt.subplots(figsize=(7, 6))
residuals_val = y_test - y_ridge_pred
ax.scatter(y_ridge_pred, residuals_val, alpha=0.35, color='#e74c3c', edgecolors='none', s=20)
ax.axhline(0, color='black', linestyle='--', linewidth=1.5)
ax.set_title('Residual Plot (Errors vs. Predicted Values)', fontweight='bold', pad=12)
ax.set_xlabel('Predicted Loan Sanction Amount (USD)')
ax.set_ylabel('Residuals (y - y_hat)')
ax.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
fig.savefig('figures/residual_plot.png', dpi=300)
fig.savefig('residual_plot.png', dpi=300)
plt.show()

# 7. Validation Curve (Training vs Validation Score vs Alpha)
alpha_values = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
train_scores, val_scores = [], []
for a in alpha_values:
    m_ridge = Ridge(alpha=a, random_state=42)
    m_ridge.fit(X_train, y_train)
    train_scores.append(m_ridge.score(X_train, y_train))
    val_scores.append(m_ridge.score(X_test, y_test))
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(alpha_values, train_scores, marker='o', label='Training R2', color='#16a085', linewidth=2)
ax.plot(alpha_values, val_scores, marker='s', label='Validation R2', color='#d35400', linewidth=2)
ax.set_xscale('log')
ax.set_xlabel('Regularization Parameter alpha (Log Scale)')
ax.set_ylabel('R2 Score')
ax.set_title('Ridge Regression: Training vs. Validation R2 Score', fontweight='bold', pad=12)
ax.legend()
ax.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
fig.savefig('figures/train_vs_val_error.png', dpi=300)
fig.savefig('train_vs_val_error.png', dpi=300)
plt.show()